# dashboard production health integration v1

目标：把 production health export 接入 dashboard export 和静态 dashboard 页面，让 production current 指针状态能进入 review 页面。

边界：notebook 只调用 `src.schedule`/`src.dashboard` 稳定接口，不直接读取 dashboard 内部散落文件。


In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import build_dashboard_view_model
from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = Path('research/dashboard_production_health_integration_v1')
CARD.mkdir(parents=True, exist_ok=True)
notifications = []
MATRIX_CARD = Path('production/paper_sim_replay_v1/matrix_card')
HEALTH_PATH = Path('research/production_registry_health_v1/production_health_export.json')
config = ModelDashboardPipelineConfig(
    name='research_dashboard_production_health_v1',
    card_dir=MATRIX_CARD,
    dashboard_output_path=CARD / 'dashboard_export.json',
    dashboard_page_output_path=CARD / 'dashboard.html',
    dashboard_page_title='SOXL Production Health Dashboard',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'random_forest_clf'),
    top_n_factors=3,
    dashboard_top_n=5,
    production_health_path=HEALTH_PATH,
    notify_details=True,
)
config


ModelDashboardPipelineConfig(name='research_dashboard_production_health_v1', card_dir=PosixPath('production/paper_sim_replay_v1/matrix_card'), dashboard_output_path=PosixPath('research/dashboard_production_health_integration_v1/dashboard_export.json'), start='2026-01-01', end='2026-06-09', train_end='2026-04-30', model_names=('ridge_mse', 'random_forest_clf'), top_n_factors=3, dashboard_top_n=5, attribution_card_dir=None, production_health_path=PosixPath('research/production_registry_health_v1/production_health_export.json'), dashboard_page_output_path=PosixPath('research/dashboard_production_health_integration_v1/dashboard.html'), dashboard_page_title='SOXL Production Health Dashboard', analysis_manifest_preflight=False, analysis_manifest_auto_backfill=False, analysis_manifest_dataset_name='schedule_dashboard_refresh', analysis_manifest_universe=('SOXL',), analysis_manifest_calendar_path='', notify_details=True, retries=0)

In [2]:
def reuse_existing_matrix_card(**kwargs):
    return {
        'status': 'reuse_existing_production_matrix_card',
        'variant_count': 0,
        'card_dir': str(kwargs['card_dir']),
    }

result = run_model_dashboard_refresh_once(
    config,
    lock_dir=CARD / 'locks',
    notify=notifications.append,
    matrix_runner=reuse_existing_matrix_card,
)
result


JobResult(name='research_dashboard_production_health_v1', status='success', payload={'matrix_card': 'production/paper_sim_replay_v1/matrix_card', 'dashboard_export': 'research/dashboard_production_health_integration_v1/dashboard_export.json', 'dashboard_schema_version': 'dashboard.model_matrix.v1', 'dashboard_artifact_id': 'matrix_card', 'dashboard_manifest_kind': 'analysis_manifest', 'dashboard_manifest_path': 'production/paper_sim_replay_v1/matrix_card/analysis_manifest.json', 'dashboard_manifest_hash': '4d16a54a928730947e024297c634659ca6ecb1f8d575073d6878666e55bf2b46', 'dashboard_manifest_producer': 'src.analysis.exports', 'dashboard_page': 'research/dashboard_production_health_integration_v1/dashboard.html', 'dashboard_page_schema_version': 'dashboard.static_page.v1', 'dashboard_view_model_schema_version': 'dashboard.view_model.v1', 'dashboard_page_model_card_count': 5, 'top_model': 'ridge_mse__ic_weighted_composite', 'attribution_card': '', 'attribution_status': 'not_provided', 'm

In [3]:
dashboard_payload = json.loads((CARD / 'dashboard_export.json').read_text(encoding='utf-8'))
view_model = build_dashboard_view_model(dashboard_payload)
health = dashboard_payload['production_health']
summary = {
    'schema_version': 'research.dashboard_production_health_integration.v1',
    'status': result.status,
    'dashboard_schema_version': dashboard_payload['schema_version'],
    'view_model_schema_version': view_model['schema_version'],
    'production_health_schema_version': health['schema_version'],
    'strategy_id': health['strategy_id'],
    'health_status': health['health_status'],
    'is_healthy': health['is_healthy'],
    'checked_path_count': health['checked_path_count'],
    'missing_required_paths': health['missing_required_paths'],
    'dashboard_contains_production_health': 'production_health' in dashboard_payload,
    'view_model_contains_production_health': 'production_health' in view_model,
    'dashboard_export': str(CARD / 'dashboard_export.json'),
    'dashboard_html': str(CARD / 'dashboard.html'),
    'notification': notifications[0] if notifications else '',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'research.dashboard_production_health_integration.v1',
 'status': 'success',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'view_model_schema_version': 'dashboard.view_model.v1',
 'production_health_schema_version': 'schedule.production_health.v1',
 'strategy_id': 'paper_sim_replay_v1',
 'health_status': 'healthy',
 'is_healthy': True,
 'checked_path_count': 7,
 'missing_required_paths': [],
 'dashboard_contains_production_health': True,
 'view_model_contains_production_health': True,
 'dashboard_export': 'research/dashboard_production_health_integration_v1/dashboard_export.json',
 'dashboard_html': 'research/dashboard_production_health_integration_v1/dashboard.html',
 'notification': 'research_dashboard_production_health_v1:success | top_model=ridge_mse__ic_weighted_composite | dashboard_manifest=analysis_manifest | attempts=1 | export=research/dashboard_production_health_integration_v1/dashboard_export.json | top_failure=return_stability_lag@stress | t

In [4]:
html = (CARD / 'dashboard.html').read_text(encoding='utf-8')
assert result.status == 'success'
assert dashboard_payload['production_health']['schema_version'] == 'schedule.production_health.v1'
assert dashboard_payload['production_health']['health_status'] == 'healthy'
assert view_model['production_health']['strategy_id'] == 'paper_sim_replay_v1'
assert 'Production Health' in html
assert 'paper_sim_replay_v1' in html
assert 'missing 0' in html
assert 'live broker off' in html
summary


{'schema_version': 'research.dashboard_production_health_integration.v1',
 'status': 'success',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'view_model_schema_version': 'dashboard.view_model.v1',
 'production_health_schema_version': 'schedule.production_health.v1',
 'strategy_id': 'paper_sim_replay_v1',
 'health_status': 'healthy',
 'is_healthy': True,
 'checked_path_count': 7,
 'missing_required_paths': [],
 'dashboard_contains_production_health': True,
 'view_model_contains_production_health': True,
 'dashboard_export': 'research/dashboard_production_health_integration_v1/dashboard_export.json',
 'dashboard_html': 'research/dashboard_production_health_integration_v1/dashboard.html',
 'notification': 'research_dashboard_production_health_v1:success | top_model=ridge_mse__ic_weighted_composite | dashboard_manifest=analysis_manifest | attempts=1 | export=research/dashboard_production_health_integration_v1/dashboard_export.json | top_failure=return_stability_lag@stress | t

## 中文结论

- 做了什么：通过 `src.schedule.run_model_dashboard_refresh_once` 调用 dashboard export，把 `research/production_registry_health_v1/production_health_export.json` 合并进 dashboard payload，并写出 `dashboard.html`。
- 核心指标：`production_health.schema_version=schedule.production_health.v1`，`strategy_id=paper_sim_replay_v1`，`health_status=healthy`，`missing_required_paths=[]`，页面包含 `Production Health` 面板。
- regime 差异：本实验不重新训练或回测；regime、window、failure mode 仍来自 production matrix card 的 analysis manifest，health 面板只补充 production current 指针状态。
- 是否晋升：否。本实验只把 production health 接入 dashboard review，不改变策略晋升状态。
- 风险：health 面板说明 manifest/sidecar 完整性，不等同于 live broker 可用或未来收益稳定；当前仍保持 paper/sim。
- 下一步：把 dashboard health 面板接入外部定时通知和页面入口，并增加异常状态截图/告警审计。
